### Knowledge Agents

Creates Agent Bricks Knowledge Assistants for every document corpus used by the
operational dashboard supervisor:

- **Inspection KA** — food safety inspection reports
- **Menu KA** — restaurant menu PDFs (items, nutrition, allergens)
- **Legal KA** — employment, liability, vendor disputes
- **Regulatory KA** — permits, fire safety, zoning, FDA
- **Audit KA** — financial, operational, food safety, supply chain audits
- **Consultancy KA** — strategy, operations, AI transformation, workforce

KAs are created in parallel; readiness polling is delegated to the downstream
`Readiness_Check` task.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Knowledge Assistant configuration

One config dict per KA. Each is created via the Knowledge Assistants REST API
(`/api/2.1/knowledge-assistants`) in two steps — create the KA, then attach a
`files` knowledge source pointing at the UC Volume — and registered in
`uc_state` for cleanup. Endpoint names are auto-generated by the API and read
back from the create response.

In [ ]:
from databricks.sdk import WorkspaceClient
import json, sys
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

# Knowledge Assistants v2.1 — endpoint names are auto-generated by Databricks
# and read back from the create response (no longer user-supplied).
API_BASE = "/api/2.1/knowledge-assistants"

# Each entry produces one KA + one "files" knowledge source attached to a UC Volume.
# `display_name` must be unique at workspace level (used to look up via List API).
# `source_display_name` shows up in the Databricks UI but does not affect routing.
KA_CONFIGS = [
    {
        "display_name": f"{CATALOG}-inspection-knowledge",
        "volume_path": f"/Volumes/{CATALOG}/food_safety/reports",
        "source_display_name": "Inspection Reports",
        "description": (
            "Answers questions about food safety inspections at Casper's Kitchens "
            "ghost kitchen locations. Covers inspection scores, violations, "
            "corrective actions, and compliance status across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Food safety inspection report PDFs for 8 ghost kitchen locations (US + EMEA). "
            "Each PDF contains the full inspection report with facility information, overall "
            "score, letter grade, violation details, corrective actions, and follow-up status."
        ),
        "instructions": (
            "You are a food safety compliance assistant for Casper's Kitchens. "
            "Always cite the specific inspection report (location and date) when answering. "
            "Be precise about violation severities (critical, major, minor), corrective "
            "actions, and deadlines. Flag any critical violations prominently."
        ),
    },
    {
        "display_name": f"{CATALOG}-menu-knowledge",
        "volume_path": f"/Volumes/{CATALOG}/menu_documents/menus",
        "source_display_name": "Menu PDFs",
        "description": (
            "Answers questions about Casper's Kitchens restaurant menus including dishes, "
            "ingredients, nutrition, allergens, and preparation details. Covers all 16 "
            "restaurant brands."
        ),
        "source_description": (
            "Restaurant menu PDFs for 16 ghost kitchen brands. Each PDF contains the full menu "
            "with item names, descriptions, prices, nutritional information (calories, protein, "
            "fat, carbs), and allergen warnings."
        ),
        "instructions": (
            "You are a restaurant menu assistant for Casper's Kitchens. "
            "Always cite which brand the information comes from. "
            "Be specific about allergen warnings and nutritional details."
        ),
    },
    {
        "display_name": f"{CATALOG}-legal",
        "volume_path": f"/Volumes/{CATALOG}/legal_complaints/documents",
        "source_display_name": "Legal Complaint Documents",
        "description": (
            "Answers questions about Casper's Kitchens legal exposure. Covers employment "
            "disputes (discrimination, wrongful termination, wage violations), customer "
            "liability claims, and vendor/contract disputes across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Legal complaint PDFs for Casper's Kitchens Inc. Each document is a confidential "
            "attorney-client privileged case file covering case number, parties, legal basis, "
            "applicable statutes, relief sought, case status, and risk assessment."
        ),
        "instructions": (
            "You are a legal intelligence assistant for Casper's Kitchens. "
            "Always cite the specific case number and location when answering. "
            "Clearly state the risk level (HIGH/MEDIUM/LOW) and amount at stake. "
            "Remind users that all legal decisions should be made with counsel. "
            "Never speculate on legal outcomes."
        ),
    },
    {
        "display_name": f"{CATALOG}-regulatory",
        "volume_path": f"/Volumes/{CATALOG}/regulatory/documents",
        "source_display_name": "Regulatory Documents",
        "description": (
            "Answers questions about Casper's Kitchens regulatory compliance status. "
            "Covers food service permits, fire safety certificates, zoning compliance, "
            "FDA registrations, and food handler certifications across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Regulatory compliance PDFs for all Casper's Kitchens ghost kitchen locations. "
            "Includes document type, issuing authority, issue date, expiry date, current status, "
            "and specific conditions and requirements from each regulatory body."
        ),
        "instructions": (
            "You are a regulatory compliance assistant. "
            "Always cite the specific document ID, issuing authority, and expiry date. "
            "Flag any permits or certificates that are expiring within 60 days or have a conditional status. "
            "Summarize compliance risk clearly and concisely."
        ),
    },
    {
        "display_name": f"{CATALOG}-audits",
        "volume_path": f"/Volumes/{CATALOG}/audits/reports",
        "source_display_name": "Audit Reports",
        "description": (
            "Answers questions about Casper's Kitchens audit findings. "
            "Covers financial statement audits, operational compliance, food safety management, "
            "and supply chain audits across all locations and quarters."
        ),
        "source_description": (
            "Independent audit reports prepared by Big 4 and mid-tier firms for Casper's Kitchens. "
            "Each report includes audit type, period, scope, auditor's opinion, individual findings "
            "(Critical/Significant/Minor/Informational) with remediation details."
        ),
        "instructions": (
            "You are an audit intelligence assistant. "
            "Always cite the specific audit ID, auditing firm, and period covered. "
            "Highlight Critical and Significant findings prominently. "
            "Summarize the auditor's opinion and any patterns across multiple audits. "
            "Be specific about financial impacts and remediation deadlines."
        ),
    },
    {
        "display_name": f"{CATALOG}-consultancy",
        "volume_path": f"/Volumes/{CATALOG}/consultancy/reports",
        "source_display_name": "Consultancy Reports",
        "description": (
            "Answers questions about strategic recommendations from management consultants. "
            "Covers market expansion, operations efficiency, AI transformation, and workforce "
            "management reports from top-tier consulting firms."
        ),
        "source_description": (
            "Management consulting reports prepared exclusively for Casper's Kitchens. "
            "Each report includes executive summary, detailed section-by-section analysis, "
            "financial projections, and specific recommendations with ROI estimates."
        ),
        "instructions": (
            "You are a strategic intelligence assistant. "
            "Always cite the consulting firm and report date. "
            "Summarize key recommendations with financial impact figures. "
            "When multiple reports address the same topic, synthesize the consensus view. "
            "Be direct and executive-ready in your summaries."
        ),
    },
]

##### Create the KAs in parallel

Looks up existing KAs in `uc_state` first to skip already-created ones, then
queries the KA API once for anything still missing. New KAs are created
concurrently. The downstream `Readiness_Check` task polls the endpoints.

In [ ]:
# Resolve already-existing KAs by display_name.
# Source 1: uc_state (cheap, scoped to this catalog).
# Source 2: workspace-wide List Knowledge Assistants API (v2.1 has this; v2.0 didn't).
# Each entry maps display_name -> {"id", "endpoint_name"}.
known_kas = {}

try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        dn = info.get("name", "")  # uc_state stores display_name under "name" for back-compat
        kid = info.get("tile_id", "")
        ep = info.get("endpoint_name", "")
        if dn and kid and dn not in known_kas:
            known_kas[dn] = {"id": kid, "endpoint_name": ep}
    print(f"uc_state: found {len(known_kas)} KAs")
except Exception as e:
    print(f"\u26a0\ufe0f uc_state lookup failed: {e}")

target_names = {cfg["display_name"] for cfg in KA_CONFIGS}
missing_names = target_names - set(known_kas)
if missing_names:
    print(f"{len(missing_names)} KAs missing from uc_state \u2014 listing workspace KAs via {API_BASE}\u2026")
    try:
        params = {}
        while True:
            resp = w.api_client.do("GET", API_BASE, query=params)
            for ka in resp.get("knowledge_assistants", []):
                dn = ka.get("display_name", "")
                if dn in target_names and dn not in known_kas:
                    known_kas[dn] = {
                        "id": ka.get("id", ""),
                        "endpoint_name": ka.get("endpoint_name", ""),
                    }
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"\u26a0\ufe0f KA list error: {e}")


def _create_one(cfg):
    display_name = cfg["display_name"]

    # Skip if a KA with this display_name already exists.
    if display_name in known_kas:
        existing = known_kas[display_name]
        print(f"[{display_name}] \u267b\ufe0f  Already exists (id={existing['id']}) \u2014 skipping create")
        return {
            "display_name": display_name,
            "id": existing["id"],
            "endpoint_name": existing["endpoint_name"],
        }

    # Step 1: create the Knowledge Assistant. v2.1 body is just metadata; the
    # endpoint_name is assigned by the API and returned in the response.
    print(f"[{display_name}] Creating KA\u2026")
    ka_body = {
        "display_name": display_name,
        "description": cfg["description"],
        "instructions": cfg["instructions"],
    }
    ka_resp = w.api_client.do("POST", API_BASE, body=ka_body) or {}
    ka_id = ka_resp.get("id", "")
    ka_endpoint = ka_resp.get("endpoint_name", "")
    if not ka_id:
        raise RuntimeError(f"[{display_name}] Create response missing id: {ka_resp}")

    # If the endpoint name wasn't populated synchronously, re-fetch.
    if not ka_endpoint:
        try:
            ka_get = w.api_client.do("GET", f"{API_BASE}/{ka_id}") or {}
            ka_endpoint = ka_get.get("endpoint_name", "")
        except Exception:
            pass

    # Step 2: attach a "files" knowledge source pointing at the UC Volume.
    # The Readiness_Check task is responsible for waiting until the source
    # finishes its initial sync.
    src_body = {
        "display_name": cfg["source_display_name"],
        "description": cfg["source_description"],
        "source_type": "files",
        "files": {"path": cfg["volume_path"]},
    }
    try:
        w.api_client.do("POST", f"{API_BASE}/{ka_id}/knowledge-sources", body=src_body)
        print(f"[{display_name}] \u2705 Created (id={ka_id}, endpoint={ka_endpoint or 'pending'})")
    except Exception as e:
        print(f"[{display_name}] \u26a0\ufe0f  KA created but source attach failed: {e}")

    return {"display_name": display_name, "id": ka_id, "endpoint_name": ka_endpoint}


created_agents = []
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {executor.submit(_create_one, cfg): cfg for cfg in KA_CONFIGS}
    for future in as_completed(futures):
        created_agents.append(future.result())

# uc_state stores: name=display_name (for routing lookups), tile_id=KA UUID
# (for delete + readiness checks), endpoint_name=auto-generated by API.
for r in created_agents:
    add(CATALOG, "knowledge_assistants", {
        "endpoint_name": r["endpoint_name"],
        "tile_id": r["id"],
        "name": r["display_name"],
    })
    print(f"   Registered: {r['display_name']} (id={r['id']}, endpoint={r['endpoint_name'] or 'pending'})")

print(f"\n\u2705 Knowledge Agents stage complete \u2014 {len(created_agents)} KAs processed")